# Extra: A Transformer as a CPU — Can a Neural Network "Run Programs"?

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/A1-Neural-CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_cpu_program.py', 'q_cpu_small_model.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

LLMs can write programs, but can they **run** them? In this unit we design a tiny 8-bit CPU (in the style of the Z80/8080) and train a Transformer to "play" that CPU:

* Input: the whole program (ROM) + the CPU's current state (registers, flags, PC, memory)
* Output: the state after executing one instruction

The Transformer has to do the **fetch** (use the PC to find the instruction to execute — exactly the kind of "table lookup" attention is good at), the **decode** and the **execute** (including 8-bit addition/subtraction with carry) all by itself.

At the end we evaluate: how accurately does it run? How long can it run without errors? Could it run an NES (Famicom) or Apple II emulator?

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## A mini CPU: TinyCPU

* Four 8-bit registers `A B C D`, flags `Z` (result is zero) and `CF` (carry/borrow)
* The program lives in a separate ROM of 16 instructions; `PC` is 0–15
* 8 bytes of data memory `M0`–`M7`
* Instruction set (`x`, `y` are registers, `n` is an 8-bit immediate):

| Instruction | Action | Instruction | Action |
|---|---|---|---|
| `NOP` | nothing | `AND y` / `OR y` / `XOR y` | `A = A op y`, sets Z, CF=0 |
| `MOV x,y` | `x = y` | `LD x,[n]` | `x = M[n]` |
| `MVI x,n` | `x = n` | `ST [n],x` | `M[n] = x` |
| `ADD y` | `A = A + y`, sets Z, CF | `JP n` | jump to n |
| `SUB y` | `A = A - y`, sets Z, CF | `JZ n` / `JNZ n` / `JC n` | conditional jump |
| `INC x` / `DEC x` | `x ± 1`, sets Z | `HALT` | stop (state unchanged) |

The state is 15 numbers: `[PC, A, B, C, D, Z, CF, M0..M7]`; each instruction is 4 numbers: `[op, x, y, n]`.

In [ ]:
OPS = ["NOP", "MOV", "MVI", "ADD", "SUB", "INC", "DEC", "AND", "OR", "XOR",
       "LD", "ST", "JP", "JZ", "JNZ", "JC", "HALT"]
OP = {name: i for i, name in enumerate(OPS)}
REGS = "ABCD"
FIELDS = ["PC", "A", "B", "C", "D", "Z", "CF"] + [f"M{i}" for i in range(8)]
N_OPS, ROM_SIZE, N_FIELDS = len(OPS), 16, len(FIELDS)

### Reference emulator

Written in PyTorch, so it can simulate a whole batch of CPUs at once (in parallel on the GPU). We use it to generate training data and to check answers.

In [ ]:
def cpu_step(prog, s):
    """prog: (N, 16, 4), s: (N, 15)  ->  state after executing one instruction (N, 15)"""
    N = s.shape[0]
    ar = torch.arange(N, device=s.device)
    pc = s[:, 0]
    op, x, y, n = prog[ar, pc].unbind(-1)
    R, Z, CF, M = s[:, 1:5], s[:, 5], s[:, 6], s[:, 7:15]
    A, rx, ry = R[:, 0], R[ar, x], R[ar, y]
    newR, newZ, newCF, newM = R.clone(), Z.clone(), CF.clone(), M.clone()
    new_pc = (pc + 1) % ROM_SIZE

    def write(mask, idx, val, zero_flag=True):
        newR[ar[mask], idx[mask]] = val[mask]
        if zero_flag:
            newZ[mask] = (val[mask] == 0).long()

    is_ = lambda name: op == OP[name]
    write(is_("MOV"), x, ry, zero_flag=False)
    write(is_("MVI"), x, n, zero_flag=False)
    zeros = torch.zeros_like(A)
    for name, t in [("ADD", A + ry), ("SUB", A - ry)]:
        m = is_(name)
        write(m, zeros, t & 255)
        newCF[m] = ((t > 255) | (t < 0)).long()[m]
    write(is_("INC"), x, (rx + 1) & 255)
    write(is_("DEC"), x, (rx - 1) & 255)
    for name, v in [("AND", A & ry), ("OR", A | ry), ("XOR", A ^ ry)]:
        m = is_(name)
        write(m, zeros, v)
        newCF[m] = 0
    write(is_("LD"), x, M[ar, n % 8], zero_flag=False)
    m = is_("ST")
    newM[ar[m], (n % 8)[m]] = rx[m]
    jump = is_("JP") | (is_("JZ") & (Z == 1)) | (is_("JNZ") & (Z == 0)) | (is_("JC") & (CF == 1))
    new_pc = torch.where(jump, n % ROM_SIZE, new_pc)
    new_pc = torch.where(is_("HALT"), pc, new_pc)
    return torch.cat([new_pc[:, None], newR, newZ[:, None], newCF[:, None], newM], dim=1)

### Assembler and display helpers

In [ ]:
def assemble(src):
    """Turn assembly source into a (16, 4) ROM, padding the rest with HALT"""
    rom = []
    for line in src.strip().split("\n"):
        line = line.split(";")[0].strip()
        if not line:
            continue
        name, *args = line.replace(",", " ").split()
        args = [a.strip("[]") for a in args]
        op, x, y, n = OP[name.upper()], 0, 0, 0
        reg = lambda a: REGS.index(a.upper())
        if name.upper() in ("MOV",):
            x, y = reg(args[0]), reg(args[1])
        elif name.upper() in ("MVI", "LD"):
            x, n = reg(args[0]), int(args[1])
        elif name.upper() == "ST":
            n, x = int(args[0]), reg(args[1])
        elif name.upper() in ("ADD", "SUB", "AND", "OR", "XOR"):
            y = reg(args[0])
        elif name.upper() in ("INC", "DEC"):
            x = reg(args[0])
        elif name.upper() in ("JP", "JZ", "JNZ", "JC"):
            n = int(args[0])
        rom.append([op, x, y, n])
    rom += [[OP["HALT"], 0, 0, 0]] * (ROM_SIZE - len(rom))
    return torch.tensor(rom)

def show_state(s):
    s = s.tolist()
    return (f"PC={s[0]:2d} A={s[1]:3d} B={s[2]:3d} C={s[3]:3d} D={s[4]:3d} Z={s[5]} CF={s[6]} "
            f"M={s[7:]}")

# Example: compute 6 x 7 with a loop and store the result in M0
MUL = """
    MVI A,0
    MVI B,7
    MVI C,6
    ADD B      ; 3: A += B
    DEC C
    JNZ 3
    ST [0],A
    HALT
"""
rom = assemble(MUL)
s = torch.zeros(1, N_FIELDS, dtype=torch.long)
for t in range(25):
    print(f"{t:2d} {OPS[rom[s[0,0], 0]]:5s}", show_state(s[0]))
    s = cpu_step(rom[None], s)

## Training data: random programs + random states

The next state depends only on the current state and the program, so we can generate an unlimited number of random (program, state) pairs and compute the answers directly on the GPU with the reference emulator.

In [ ]:
def random_batch(N, device=device):
    op = torch.randint(0, N_OPS, (N, ROM_SIZE), device=device)
    x = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    y = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    n = torch.randint(0, 256, (N, ROM_SIZE), device=device)
    # Keep most jump targets and memory addresses in a sensible range
    small = torch.randint(0, 16, (N, ROM_SIZE), device=device)
    n = torch.where(torch.rand(N, ROM_SIZE, device=device) < 0.5, small, n)
    prog = torch.stack([op, x, y, n], dim=-1)
    s = torch.cat([torch.randint(0, ROM_SIZE, (N, 1), device=device),
                   torch.randint(0, 256, (N, 4), device=device),
                   torch.randint(0, 2, (N, 2), device=device),
                   torch.randint(0, 256, (N, 8), device=device)], dim=1)
    # Set some registers to small numbers, 0 or 255 to get more carry and zero cases
    special = torch.tensor([0, 1, 2, 254, 255], device=device)
    mask = torch.rand(N, 4, device=device) < 0.2
    s[:, 1:5] = torch.where(mask, special[torch.randint(0, 5, (N, 4), device=device)], s[:, 1:5])
    return prog, s, cpu_step(prog, s)

prog, s, s_next = random_batch(3)
print(prog[0, :4])
print(s[0], "\n", s_next[0])

Completely random states look very different from the states a real program runs into: real programs usually start from an all-zero state, are padded with `HALT`, and use smaller numbers.
So we add a second kind of "realistic program" data: random program length with the rest padded with `HALT`, start from the all-zero state, run a random number of steps, then use the next step as the question.
Training uses half of each kind.

In [ ]:
HALT_INS = torch.tensor([OP["HALT"], 0, 0, 0], device=device)

def program_batch(N, device=device, max_warmup=24):
    prog, _, _ = random_batch(N, device)
    length = torch.randint(3, ROM_SIZE + 1, (N, 1), device=device)
    prog[torch.arange(ROM_SIZE, device=device)[None] >= length] = HALT_INS
    s = torch.zeros(N, N_FIELDS, dtype=torch.long, device=device)
    for _ in range(np.random.randint(0, max_warmup)):
        s = cpu_step(prog, s)
    return prog, s, cpu_step(prog, s)

def mixed_batch(N):
    a, b = random_batch(N // 2), program_batch(N - N // 2)
    return tuple(torch.cat([x, y]) for x, y in zip(a, b))

## Model: a Transformer CPU

The input is 16 "instruction tokens" + 15 "state tokens", 31 tokens in total:

* **Numbers are represented in binary**: a byte is split into 8 bits and mapped to a vector by a `Linear`; the output also predicts 8 bits. This makes the carry rule of addition much easier to learn (far easier than treating 0–255 as 256 unrelated symbols)
* Instruction token = opcode embedding + register embeddings + immediate-value bits + **address embedding** (so attention can use the PC to find the instruction)
* State token = value bits + field embedding (is this the PC, A, or M3, ...?)
* A bidirectional (non-causal) Transformer encoder: the next values of all 15 state fields are predicted **at the same time**

In [ ]:
def to_bits(v):
    return ((v[..., None] >> torch.arange(8, device=v.device)) & 1).float()

def from_bits(logits):
    return ((logits > 0).long() << torch.arange(8, device=logits.device)).sum(-1)

class NeuralCPU(nn.Module):
    def __init__(self, d=256, n_layers=6, n_heads=8):
        super().__init__()
        self.op_emb = nn.Embedding(N_OPS, d)
        self.x_emb = nn.Embedding(4, d)
        self.y_emb = nn.Embedding(4, d)
        self.addr_emb = nn.Embedding(ROM_SIZE, d)
        self.field_emb = nn.Embedding(N_FIELDS, d)
        self.bits_in = nn.Linear(8, d)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d)
        self.bits_out = nn.Linear(d, 8)

    def forward(self, prog, s):
        op, x, y, n = prog.unbind(-1)
        addr = torch.arange(ROM_SIZE, device=prog.device)
        fields = torch.arange(N_FIELDS, device=prog.device)
        prog_tok = self.op_emb(op) + self.x_emb(x) + self.y_emb(y) + self.bits_in(to_bits(n)) + self.addr_emb(addr)
        state_tok = self.bits_in(to_bits(s)) + self.field_emb(fields)
        h = self.encoder(torch.cat([prog_tok, state_tok], dim=1))
        return self.bits_out(self.norm(h[:, ROM_SIZE:]))       # (N, 15, 8) bit logits

    @torch.no_grad()
    def step(self, prog, s):
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            return from_bits(self(prog, s).float())

model = NeuralCPU().to(device)
print(f"number of parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Training

Every step generates a brand-new batch, so the model cannot overfit. The loss is the binary cross entropy of each bit.
It takes about 35 minutes on a T4 (accuracy keeps improving until the end). You can try a smaller `TRAIN_STEPS` first, but accuracy will be noticeably lower: with 8000 steps it stays around 96%, which is not enough to run real programs.

In [ ]:
@torch.no_grad()
def evaluate(model, N=20000, batch_fn=None, chunk=4096):
    model.eval()
    prog, s, target = (batch_fn or mixed_batch)(N)
    pred = torch.cat([model.step(prog[i:i + chunk], s[i:i + chunk]) for i in range(0, N, chunk)])
    ok = (pred == target).all(1)
    op = prog[torch.arange(N, device=device), s[:, 0], 0]
    per_op = {OPS[i]: ok[op == i].float().mean().item() for i in range(N_OPS)}
    return ok.float().mean().item(), per_op

def train(model, steps, batch_size=512, lr=1e-3, log_every=1000):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps, pct_start=0.05)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    t0 = time.perf_counter()
    for step in range(1, steps + 1):
        model.train()
        prog, s, target = mixed_batch(batch_size)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(prog, s)
        loss = F.binary_cross_entropy_with_logits(logits.float(), to_bits(target))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if step % log_every == 0 or step == steps:
            acc, _ = evaluate(model, 4096)
            print(f"step {step:5d}  loss {loss.item():.5f}  single-step exact accuracy {acc:.4f}  {time.perf_counter()-t0:.0f}s")

TRAIN_STEPS = 15000
train(model, TRAIN_STEPS)

## Evaluation 1: single-step accuracy

Random program, random state, execute one instruction: it counts as correct only if **all 15 fields are right**.

In [ ]:
acc, per_op = evaluate(model, 50000)
print(f"overall single-step accuracy: {acc:.5f}  (error rate {1-acc:.2e})")
print(f"  random states: {evaluate(model, 20000, random_batch)[0]:.5f}   realistic programs: {evaluate(model, 20000, program_batch)[0]:.5f}")
for name, a in sorted(per_op.items(), key=lambda kv: kv[1]):
    print(f"  {name:5s} {a:.5f}")

## Evaluation 2: actually running programs

Let the neural network execute step by step on its own (each step's output becomes the next step's input, **with no more correct answers given**) and compare with the reference emulator.

In [ ]:
def run(prog, s, steps, stepper):
    traj = [s]
    for _ in range(steps):
        s = stepper(prog, s)
        traj.append(s)
    return torch.stack(traj, dim=1)   # (N, steps+1, 15)

def compare(src, steps=100, name=""):
    rom = assemble(src)[None].to(device)
    s0 = torch.zeros(1, N_FIELDS, dtype=torch.long, device=device)
    ref = run(rom, s0, steps, cpu_step)[0]
    neu = run(rom, s0, steps, model.step)[0]
    diff = (ref != neu).any(1).nonzero()
    first = diff[0].item() if len(diff) else None
    print(f"{name}: reference   {show_state(ref[-1])}")
    print(f"{' ' * len(name)}  neural CPU  {show_state(neu[-1])}")
    print("  ✅ all steps match" if first is None else f"  ❌ diverges at step {first}")

FIB = """
    MVI A,1
    MVI B,0
    MVI D,10
    MOV C,A     ; 3
    ADD B
    MOV B,C
    DEC D
    JNZ 3
    ST [1],A
    HALT
"""
SUM = """
    MVI A,0
    MVI B,20
    ADD B       ; 2: A += B
    DEC B
    JNZ 2
    ST [2],A
    HALT
"""
compare(MUL, 40, "6x7")
compare(FIB, 60, "fib")
compare(SUM, 80, "1+..+20")

## Evaluation 3: errors accumulate

Even with 99.9% single-step accuracy, the probability that all $k$ steps are correct is about $0.999^k$. A real computer executes millions of instructions per second, and **a single wrong bit can crash the program**.

Run 2000 randomly generated programs (starting from the all-zero state) for 100 steps each, and look at the fraction that is "still exactly correct up to step k":

In [ ]:
N, K = 2000, 100
prog, s0, _ = program_batch(N, max_warmup=1)
ref = run(prog, s0, K, cpu_step)
neu = run(prog, s0, K, model.step)
ok_until = (ref == neu).all(-1).cumprod(1).float().mean(0).cpu()
p1 = ok_until[1].item()
plt.plot(ok_until, label="neural CPU")
plt.plot([p1 ** k for k in range(K + 1)], "--", label=f"{p1:.4f}^k")
plt.xlabel("steps"); plt.ylabel("fraction still exactly correct"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"fraction still exactly correct after 100 steps: {ok_until[-1]:.3f}")

## Q
Write your own program (e.g. compute $2^7$, take the maximum of two numbers, or compute the quotient of 100 divided by 7), run it on both the reference emulator and the neural CPU, and compare.

In [ ]:
# %load q_cpu_program.py
%run -i q_cpu_program.py

## Q
If the model has only **2 layers**, which instructions are hardest to learn? Why? (Hint: the carry in 8-bit addition may have to propagate all the way from bit 0 to bit 7.)

In [ ]:
# %load q_cpu_small_model.py
%run -i q_cpu_small_model.py

## Evaluation 4: speed — could it run an NES?

In [ ]:
rom = assemble(SUM)[None].to(device)
s = torch.zeros(1, N_FIELDS, dtype=torch.long, device=device)
model.step(rom, s)  # warm up
if device == "cuda": torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(200):
    s = model.step(rom, s)
if device == "cuda": torch.cuda.synchronize()
single = 200 / (time.perf_counter() - t0)

B = 8192
progB, sB, _ = random_batch(B)
model.step(progB, sB)
if device == "cuda": torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(20):
    sB = model.step(progB, sB)
if device == "cuda": torch.cuda.synchronize()
batched = 20 * B / (time.perf_counter() - t0)

nes = 1.79e6 / 3.5    # the NES's 6502: 1.79 MHz, about 3.5 cycles per instruction on average
print(f"neural CPU (single)        : {single:10,.0f} instructions/s")
print(f"neural CPU ({B} in parallel): {batched:10,.0f} instructions/s (total)")
print(f"NES 6502                   : {nes:10,.0f} instructions/s")
print(f"A single neural CPU is {nes / single:,.0f}x slower than the NES; one NES frame (1/60 s) is about {nes/60:,.0f} instructions, "
      f"which takes the neural CPU {nes / 60 / single:,.0f} s")

## Conclusion: can a Transformer run a Z80, an NES, or an Apple II?

| | TinyCPU (this unit) | Z80 / 6502 | NES / Apple II |
|---|---|---|---|
| Registers | four 8-bit | 7–14, including 16-bit ones | 6502: A, X, Y, SP, P, PC |
| Memory | 8 bytes + 16 instructions, all in the context | 64KB address space | NES: 2KB RAM + 32KB+ cartridge + PPU graphics chip; Apple II: 48KB RAM with memory-mapped display |
| Instructions | 17 | Z80 about 700 (with prefixes), 6502 151 | also need to emulate the PPU, sound, timing, mappers |
| Feasibility | ✅ single-step accuracy can approach 100% | a single step can be learned, but memory does not fit in the context; a separate memory read/write mechanism is needed | ❌ tens of thousands of times too slow, and one wrong bit crashes the game |

Key takeaways
* A Transformer can learn **single-step** computation very well; the hard part is **precise execution over long runs**: errors accumulate. This is one reason LLMs make mistakes in long chains of exact computation
* That is why, for exact computation, LLMs usually **call tools** (e.g. write Python code and run it; see tool calling in 04-LLM-Apps) instead of doing "mental arithmetic"
* Researchers have also **hand-designed weights** that make a Transformer execute programs exactly (e.g. *Looped Transformers as Programmable Computers*, 2023; DeepMind's *Tracr* compiles programs into Transformer weights). This never makes mistakes, but the weights are not learned
* If the goal is something that "looks like playing the game", see **world models**: e.g. GameNGen (2024) generates DOOM frames one by one with a diffusion model, and Oasis (2024) generates Minecraft. They imitate the screen rather than executing the game code, and training them takes far more compute than a T4